In [129]:
import re
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import matplotlib.cm as cm
import numpy as np
import contextily as ctx
import geopandas as gpd
import folium
from shapely.geometry import Point, LineString
from matplotlib.patches import Circle
import warnings
warnings.filterwarnings('ignore')

# 讀檔
with open("all testing progress/2.txt", "r", encoding="utf-8") as f:
    lines = f.readlines()


In [131]:
# Regex
pattern = re.compile(
    r'^(?P<date>\d{2}-\d{2} \d{2}:\d{2}:\d{2}\.\d{3}).* \[(?P<object>.*?Variant\(Clone\))\]\s+location → '
    r'Lon: (?P<lon>[-\d\.]+)°, Lat: (?P<lat>[-\d\.]+)°, H: (?P<height>[\d\.]+) m'
)
destroy_pattern = re.compile(r"\[(.*?Variant\(Clone\))\].*Destroy")

records = []
destroy_events = []

# 暫存每個物件最後座標
last_position = {}

for line in lines:
    m = pattern.search(line)
    if m:
        d = pd.to_datetime(m.group("date"), format="%m-%d %H:%M:%S.%f")
        obj = m.group("object")
        lon, lat, height = float(m.group("lon")), float(m.group("lat")), float(m.group("height"))

        records.append({"time": d, "object": obj, "lon": lon, "lat": lat, "height": height})

        # 更新最後座標
        last_position[obj] = (lat, lon)
    else:
        m2 = destroy_pattern.search(line)
        if m2:
            obj = m2.group(1)
            if obj in last_position:  # ✅ Destroy → 用前一筆座標
                destroy_events.append({"object": obj, "lat": last_position[obj][0], "lon": last_position[obj][1]})

df = pd.DataFrame(records).sort_values("time")
center = [df["lat"].mean(), df["lon"].mean()]

# 顏色與標籤
object_colormaps = {
    "OneVariant(Clone)": "Blues",
    "TwoVariant(Clone)": "Reds",
    "ThreeVariant(Clone)": "Greens",
    "FourVariant(Clone)": "Oranges",
    "FiveVariant(Clone)": "Purples",  # 改為更有對比的配色
    "SixVariant(Clone)": "pink",
    "DebugCubeVariant(Clone)": "magma",  # 改為變化較小的配色
}

object_labels = {
    "OneVariant(Clone)": "1",
    "TwoVariant(Clone)": "2",
    "ThreeVariant(Clone)": "3",
    "FourVariant(Clone)": "4",
    "FiveVariant(Clone)": "5",
    "SixVariant(Clone)": "6",
    "DebugCubeVariant(Clone)": "D",
}

# ✅ 設定空底圖，讓我們可以自定義多種底圖
m = folium.Map(location=center, zoom_start=16, tiles=None)

# ✅ 衛星圖底圖（Esri World Imagery）
folium.TileLayer(
    tiles="https://server.arcgisonline.com/ArcGIS/rest/services/World_Imagery/MapServer/tile/{z}/{y}/{x}",
    attr="Esri",
    name="Esri Satellite",
    overlay=False,
    control=True
).add_to(m)



# ✅ CartoDB Positron 底圖（原本的）
folium.TileLayer(
    tiles="https://{s}.basemaps.cartocdn.com/dark_all/{z}/{x}/{y}{r}.png",
    attr="CartoDB",
    name="CartoDB Dark Matter",
    overlay=False,
    control=True
).add_to(m)

# ✅ CartoDB Positron 底圖（原本的）
folium.TileLayer(
    tiles="https://{s}.basemaps.cartocdn.com/light_all/{z}/{x}/{y}{r}.png",
    attr="CartoDB",
    name="CartoDB Positron",
    overlay=False,
    control=True
).add_to(m)

# ✅ 指定順序：123456Debug
object_order = [
    "OneVariant(Clone)",
    "TwoVariant(Clone)",
    "ThreeVariant(Clone)",
    "FourVariant(Clone)",
    "FiveVariant(Clone)",
    "SixVariant(Clone)",
    "DebugCubeVariant(Clone)",
]

# 畫軌跡
for obj in object_order:
    if obj not in df["object"].values and all(e["object"] != obj for e in destroy_events):
        continue  # 完全沒資料略過

    g = df[df["object"] == obj]
    trajectory = g[["lat", "lon"]].values.tolist()

    layer = folium.FeatureGroup(name=obj)

    # ✅ 畫軌跡（有兩點以上才畫線）
    if len(trajectory) > 1:
        cmap_name = object_colormaps.get(obj, "Greys")
        cmap_obj = matplotlib.colormaps[cmap_name]
        if cmap_name in ["pink", "Blues", "Reds", "Greens", "Oranges", "plasma"]:
            cmap = cmap_obj.reversed()  
        else:
            cmap = cmap_obj

        n_points = len(trajectory)
        
        # 🌟 改進的顏色與視覺效果設定 - 避免過淡的顏色
        for j in range(len(trajectory) - 1):
            # 🌟 顏色漸變：限制最小值避免過淡，從0.3到1.0的範圍
            color_intensity = 0.3 + (j / (n_points - 1)) * 0.7 if n_points > 1 else 0.8
            color = mcolors.to_hex(cmap(color_intensity))
            
            # 🌟 透明度固定為較高值，避免過於透明
            opacity = 0.8  # 固定透明度，不再漸變
            
            folium.PolyLine(
                locations=[trajectory[j], trajectory[j + 1]],
                color=color,
                weight=4,  # 固定線寬
                opacity=opacity,
            ).add_to(layer)

    # ✅ 起始與終點標記
    if len(g) > 0:
        first_point, last_point = g.iloc[0], g.iloc[-1]
        label = object_labels.get(obj, "?")

        if len(g) == 1:
            # 只有一點時，只顯示ending標記
            folium.Marker(
                location=[first_point["lat"], first_point["lon"]],
                popup=f"{obj} Point<br>Time: {first_point['time']}<br>Height: {first_point['height']:.1f}m",
                icon=folium.DivIcon(
                    html=f'<div style="font-size:14px; color:white; background-color:red; '
                         f'border: 2px solid white; border-radius:50%; width:24px; height:24px; '
                         f'text-align:center; line-height:20px; font-weight:bold;">{label}</div>'
                ),
            ).add_to(layer)
        else:
            # 多於一點時，顯示起始和終點標記
            # 🌟 改進起始點標記樣式
            folium.Marker(
                location=[first_point["lat"], first_point["lon"]],
                popup=f"{obj} Starting Point<br>Time: {first_point['time']}<br>Height: {first_point['height']:.1f}m",
                icon=folium.DivIcon(
                    html=f'<div style="font-size:14px; color:white; background-color:gray; '
                         f'border: 2px solid white; border-radius:50%; width:24px; height:24px; '
                         f'text-align:center; line-height:20px; font-weight:bold;">{label}</div>'
                ),
            ).add_to(layer)

            # 🌟 改進終點標記樣式
            folium.Marker(
                location=[last_point["lat"], last_point["lon"]],
                popup=f"{obj} Ending Point<br>Time: {last_point['time']}<br>Height: {last_point['height']:.1f}m",
                icon=folium.DivIcon(
                    html=f'<div style="font-size:14px; color:white; background-color:red; '
                         f'border: 2px solid white; border-radius:50%; width:24px; height:24px; '
                         f'text-align:center; line-height:20px; font-weight:bold;">{label}</div>'
                ),
            ).add_to(layer)

    # ✅ Destroy 標記（在上一筆座標）
    for e in destroy_events:
        if e["object"] == obj:
            label = object_labels.get(obj, "?")
            folium.Marker(
                location=[e["lat"], e["lon"]],
                popup=f"{obj} Destroy",
                icon=folium.DivIcon(
                    html=f'<div style="font-size:14px; color:white; background-color:white; '
                         f'border: 2px solid white; border-radius:50%; width:24px; height:24px; '
                         f'text-align:center; line-height:20px; font-weight:bold;">🗑️</div>'
                ),
            ).add_to(layer)

    layer.add_to(m)


def dms_to_dd(d, m, s, ref):
    dd = d + m/60 + s/3600
    if ref in ['S', 'W']:
        dd *= -1
    return dd

# 🔹 定義三組座標 (轉換後)
points_coords = [
    (dms_to_dd(51,32,25,'N'), dms_to_dd(0,0,42,'W')),
    (dms_to_dd(51,32,14,'N'), dms_to_dd(0,1,5,'W')),
    (dms_to_dd(51,32,29,'N'), dms_to_dd(0,1,2,'W')),
    (dms_to_dd(51,32,47,'N'), dms_to_dd(0,0,53,'W')),
    (dms_to_dd(51,32,55,'N'), dms_to_dd(0,1,6,'W')),
]

birds_coords = [
    (dms_to_dd(51,32,24,'N'), dms_to_dd(0,0,46,'W')),
    (dms_to_dd(51,32,16,'N'), dms_to_dd(0,0,52,'W')),
    (dms_to_dd(51,32,41,'N'), dms_to_dd(0,1,5,'W')),
    (dms_to_dd(51,32,59,'N'), dms_to_dd(0,1,16,'W')),
]

bike_coords = [
    (dms_to_dd(51,32,18,'N'), dms_to_dd(0,0,42,'W')),
    (dms_to_dd(51,32,29,'N'), dms_to_dd(0,0,36,'W')),
    (dms_to_dd(51,32,26,'N'), dms_to_dd(0,1,15,'W')),
]

# 🔹 建立三個圖層
points_layer = folium.FeatureGroup(name="📍 Points")
birds_layer = folium.FeatureGroup(name="🐦 Birds") 
bike_layer = folium.FeatureGroup(name="🚴 Bikes")

# 🔹 points
for i, (lat, lon) in enumerate(points_coords, 1):
    folium.Marker(
        location=[lat, lon],
        popup=f"Point {i}",
        icon=folium.Icon(color="green", icon="map-marker"),
    ).add_to(points_layer)

# 🔹 birds
for i, (lat, lon) in enumerate(birds_coords, 1):
    folium.Marker(
        location=[lat, lon],
        popup=f"Bird {i}",
        icon=folium.Icon(color="orange", icon="star", prefix="fa"),
    ).add_to(birds_layer)

# 🔹 bike
for i, (lat, lon) in enumerate(bike_coords, 1):
    folium.Marker(
        location=[lat, lon],
        popup=f"Bike {i}",
        icon=folium.Icon(color="blue", icon="bicycle", prefix="fa"),
    ).add_to(bike_layer)

# 🔹 把圖層加到地圖
points_layer.add_to(m)
birds_layer.add_to(m)
bike_layer.add_to(m)

# 🔹 更新圖層控制
folium.LayerControl(collapsed=False).add_to(m)


# 🌟 加入全螢幕選項
from folium.plugins import Fullscreen
Fullscreen().add_to(m)


# 🌟 新增圖例說明
legend_html = '''
<div style="position: fixed; 
     bottom: 50px; left: 50px; width: 200px; height: 150px; 
     background-color: white; border: 2px solid grey; z-index:9999; 
     font-size: 12px; padding: 10px;">
<p><strong>Legend</strong></p>
<p>⚫ Starting | 🔴 Ending</p>
<p>🗑️ Destroy</p>
<p>Line：bright → dark </p>
<p>by time sequence</p>
</div>
'''
m.get_root().html.add_child(folium.Element(legend_html))

m

In [132]:
# Regex
pattern = re.compile(
    r'^(?P<date>\d{2}-\d{2} \d{2}:\d{2}:\d{2}\.\d{3}).* \[(?P<object>.*?Variant\(Clone\))\]\s+location → '
    r'Lon: (?P<lon>[-\d\.]+)°, Lat: (?P<lat>[-\d\.]+)°, H: (?P<height>[\d\.]+) m'
)
destroy_pattern = re.compile(r"\[(.*?Variant\(Clone\))\].*Destroy")

records = []
destroy_events = []

# 暫存每個物件最後座標
last_position = {}

for line in lines:
    m = pattern.search(line)
    if m:
        d = pd.to_datetime(m.group("date"), format="%m-%d %H:%M:%S.%f")
        obj = m.group("object")
        lon, lat, height = float(m.group("lon")), float(m.group("lat")), float(m.group("height"))

        records.append({"time": d, "object": obj, "lon": lon, "lat": lat, "height": height})

        # 更新最後座標
        last_position[obj] = (lat, lon)
    else:
        m2 = destroy_pattern.search(line)
        if m2:
            obj = m2.group(1)
            if obj in last_position:  # ✅ Destroy → 用前一筆座標
                destroy_events.append({"object": obj, "lat": last_position[obj][0], "lon": last_position[obj][1]})

df = pd.DataFrame(records).sort_values("time")
center = [df["lat"].mean(), df["lon"].mean()]

# 顏色與標籤
object_colormaps = {
    "OneVariant(Clone)": "Blues",
    "TwoVariant(Clone)": "Reds",
    "ThreeVariant(Clone)": "Greens",
    "FourVariant(Clone)": "Oranges",
    "FiveVariant(Clone)": "Purples",  # 改為更有對比的配色
    "SixVariant(Clone)": "pink",
    "DebugCubeVariant(Clone)": "magma",  # 改為變化較小的配色
}

object_labels = {
    "OneVariant(Clone)": "1",
    "TwoVariant(Clone)": "2",
    "ThreeVariant(Clone)": "3",
    "FourVariant(Clone)": "4",
    "FiveVariant(Clone)": "5",
    "SixVariant(Clone)": "6",
    "DebugCubeVariant(Clone)": "D",
}

# ✅ 設定空底圖，讓我們可以自定義多種底圖
m = folium.Map(location=center, zoom_start=16, tiles=None)

# ✅ 衛星圖底圖（Esri World Imagery）
folium.TileLayer(
    tiles="https://server.arcgisonline.com/ArcGIS/rest/services/World_Imagery/MapServer/tile/{z}/{y}/{x}",
    attr="Esri",
    name="Esri Satellite",
    overlay=False,
    control=True
).add_to(m)

# ✅ CartoDB Dark Matter 底圖
folium.TileLayer(
    tiles="https://{s}.basemaps.cartocdn.com/dark_all/{z}/{x}/{y}{r}.png",
    attr="CartoDB",
    name="CartoDB Dark Matter",
    overlay=False,
    control=True
).add_to(m)

# ✅ CartoDB Positron 底圖
folium.TileLayer(
    tiles="https://{s}.basemaps.cartocdn.com/light_all/{z}/{x}/{y}{r}.png",
    attr="CartoDB",
    name="CartoDB Positron",
    overlay=False,
    control=True
).add_to(m)

# ✅ 指定順序：123456Debug
object_order = [
    "OneVariant(Clone)",
    "TwoVariant(Clone)",
    "ThreeVariant(Clone)",
    "FourVariant(Clone)",
    "FiveVariant(Clone)",
    "SixVariant(Clone)",
    "DebugCubeVariant(Clone)",
]

# 畫軌跡
for obj in object_order:
    if obj not in df["object"].values and all(e["object"] != obj for e in destroy_events):
        continue  # 完全沒資料略過

    g = df[df["object"] == obj]
    trajectory = g[["lat", "lon"]].values.tolist()

    layer = folium.FeatureGroup(name=obj)

    # ✅ 畫軌跡（有兩點以上才畫線）
    if len(trajectory) > 1:
        cmap_name = object_colormaps.get(obj, "Greys")
        cmap_obj = matplotlib.colormaps[cmap_name]
        if cmap_name in ["pink", "Blues", "Reds", "Greens", "Oranges", "plasma"]:
            cmap = cmap_obj.reversed()  
        else:
            cmap = cmap_obj

        n_points = len(trajectory)
        
        # 🌟 改進的顏色與視覺效果設定 - 避免過淡的顏色
        for j in range(len(trajectory) - 1):
            # 🌟 顏色漸變：限制最小值避免過淡，從0.3到1.0的範圍
            color_intensity = 0.3 + (j / (n_points - 1)) * 0.7 if n_points > 1 else 0.8
            color = mcolors.to_hex(cmap(color_intensity))
            
            # 🌟 透明度固定為較高值，避免過於透明
            opacity = 0.8  # 固定透明度，不再漸變
            
            folium.PolyLine(
                locations=[trajectory[j], trajectory[j + 1]],
                color=color,
                weight=4,  # 固定線寬
                opacity=opacity,
            ).add_to(layer)

    # ✅ 起始與終點標記
    if len(g) > 0:
        first_point, last_point = g.iloc[0], g.iloc[-1]
        label = object_labels.get(obj, "?")

        if len(g) == 1:
            # 只有一點時，只顯示ending標記
            folium.Marker(
                location=[first_point["lat"], first_point["lon"]],
                popup=f"{obj} Point<br>Time: {first_point['time']}<br>Height: {first_point['height']:.1f}m",
                icon=folium.DivIcon(
                    html=f'<div style="font-size:14px; color:white; background-color:red; '
                         f'border: 2px solid white; border-radius:50%; width:24px; height:24px; '
                         f'text-align:center; line-height:20px; font-weight:bold;">{label}</div>'
                ),
            ).add_to(layer)
        else:
            # 多於一點時，顯示起始和終點標記
            # 🌟 改進起始點標記樣式
            folium.Marker(
                location=[first_point["lat"], first_point["lon"]],
                popup=f"{obj} Starting Point<br>Time: {first_point['time']}<br>Height: {first_point['height']:.1f}m",
                icon=folium.DivIcon(
                    html=f'<div style="font-size:14px; color:white; background-color:gray; '
                         f'border: 2px solid white; border-radius:50%; width:24px; height:24px; '
                         f'text-align:center; line-height:20px; font-weight:bold;">{label}</div>'
                ),
            ).add_to(layer)

            # 🌟 改進終點標記樣式
            folium.Marker(
                location=[last_point["lat"], last_point["lon"]],
                popup=f"{obj} Ending Point<br>Time: {last_point['time']}<br>Height: {last_point['height']:.1f}m",
                icon=folium.DivIcon(
                    html=f'<div style="font-size:14px; color:white; background-color:red; '
                         f'border: 2px solid white; border-radius:50%; width:24px; height:24px; '
                         f'text-align:center; line-height:20px; font-weight:bold;">{label}</div>'
                ),
            ).add_to(layer)

    # ✅ Destroy 標記（在上一筆座標）
    for e in destroy_events:
        if e["object"] == obj:
            label = object_labels.get(obj, "?")
            folium.Marker(
                location=[e["lat"], e["lon"]],
                popup=f"{obj} Destroy",
                icon=folium.DivIcon(
                    html=f'<div style="font-size:14px; color:white; background-color:white; '
                         f'border: 2px solid white; border-radius:50%; width:24px; height:24px; '
                         f'text-align:center; line-height:20px; font-weight:bold;">🗑️</div>'
                ),
            ).add_to(layer)

    layer.add_to(m)


def dms_to_dd(d, m, s, ref):
    dd = d + m/60 + s/3600
    if ref in ['S', 'W']:
        dd *= -1
    return dd

# 🔹 定義三組座標 (轉換後)
points_coords = [
    (dms_to_dd(51,32,25,'N'), dms_to_dd(0,0,42,'W')),
    (dms_to_dd(51,32,14,'N'), dms_to_dd(0,1,5,'W')),
    (dms_to_dd(51,32,29,'N'), dms_to_dd(0,1,2,'W')),
    (dms_to_dd(51,32,47,'N'), dms_to_dd(0,0,53,'W')),
    (dms_to_dd(51,32,55,'N'), dms_to_dd(0,1,6,'W')),
]

birds_coords = [
    (dms_to_dd(51,32,24,'N'), dms_to_dd(0,0,46,'W')),
    (dms_to_dd(51,32,16,'N'), dms_to_dd(0,0,52,'W')),
    (dms_to_dd(51,32,41,'N'), dms_to_dd(0,1,5,'W')),
    (dms_to_dd(51,32,59,'N'), dms_to_dd(0,1,16,'W')),
]

bike_coords = [
    (dms_to_dd(51,32,18,'N'), dms_to_dd(0,0,42,'W')),
    (dms_to_dd(51,32,29,'N'), dms_to_dd(0,0,36,'W')),
    (dms_to_dd(51,32,26,'N'), dms_to_dd(0,1,15,'W')),
]

# 🔹 建立三個圖層
points_layer = folium.FeatureGroup(name="📍 Points")
birds_layer = folium.FeatureGroup(name="🐦 Birds") 
bike_layer = folium.FeatureGroup(name="🚴 Bikes")

# 🔹 points
for i, (lat, lon) in enumerate(points_coords, 1):
    folium.Marker(
        location=[lat, lon],
        popup=f"Point {i}",
        icon=folium.Icon(color="green", icon="map-marker"),
    ).add_to(points_layer)

# 🔹 birds
for i, (lat, lon) in enumerate(birds_coords, 1):
    folium.Marker(
        location=[lat, lon],
        popup=f"Bird {i}",
        icon=folium.Icon(color="orange", icon="star", prefix="fa"),
    ).add_to(birds_layer)

# 🔹 bike
for i, (lat, lon) in enumerate(bike_coords, 1):
    folium.Marker(
        location=[lat, lon],
        popup=f"Bike {i}",
        icon=folium.Icon(color="blue", icon="bicycle", prefix="fa"),
    ).add_to(bike_layer)

# 🔹 把圖層加到地圖
points_layer.add_to(m)
birds_layer.add_to(m)
bike_layer.add_to(m)

# 🔹 更新圖層控制
# folium.LayerControl(collapsed=False).add_to(m)

# 🌟 加入全螢幕選項
# from folium.plugins import Fullscreen
# Fullscreen().add_to(m)

# 🌟 加入比例尺
from folium.plugins import MeasureControl
MeasureControl().add_to(m)

# 🌟 加入詳細的比例尺顯示
scale_html = '''
<div style="position: fixed; 
     bottom: 30px; right: 30px; width: 300px; height: 80px; 
     background-color: rgba(255,255,255,0.9); border: 2px solid black; z-index:9999; 
     font-size: 14px; padding: 15px; font-family: Arial;">
<div style="text-align: center; font-weight: bold; margin-bottom: 10px; font-size: 16px;">Scale</div>

<!-- 主要比例尺 -->
<div style="position: relative; height: 25px; margin-bottom: 8px;">
  <div style="border-bottom: 4px solid black; width: 250px; position: absolute; bottom: 0;"></div>
  <div style="border-left: 3px solid black; height: 12px; position: absolute; left: 0; bottom: 0;"></div>
  <div style="border-left: 3px solid black; height: 12px; position: absolute; left: 62.5px; bottom: 0;"></div>
  <div style="border-left: 3px solid black; height: 12px; position: absolute; left: 125px; bottom: 0;"></div>
  <div style="border-left: 3px solid black; height: 12px; position: absolute; left: 187.5px; bottom: 0;"></div>
  <div style="border-left: 3px solid black; height: 12px; position: absolute; left: 250px; bottom: 0;"></div>
</div>

<!-- 標籤 -->
<div style="position: relative; height: 20px; font-size: 12px; font-weight: bold;">
  <span style="position: absolute; left: -5px;">0</span>
  <span style="position: absolute; left: 50px;">25m</span>
  <span style="position: absolute; left: 115px;">50m</span>
  <span style="position: absolute; left: 175px;">75m</span>
  <span style="position: absolute; left: 235px;">100m</span>
</div>
</div>
'''
m.get_root().html.add_child(folium.Element(scale_html))

# 🌟 新增圖例說明
legend_html = '''
<div style="position: fixed; 
     bottom: 50px; left: 50px; width: 200px; height: 150px; 
     background-color: white; border: 2px solid grey; z-index:9999; 
     font-size: 12px; padding: 10px;">
<p><strong>Legend</strong></p>
<p>⚫ Starting | 🔴 Ending</p>
<p>🗑️ Destroy</p>
<p>Line：bright → dark </p>
<p>by time sequence</p>
</div>
'''
m.get_root().html.add_child(folium.Element(legend_html))

# 🌟 儲存為HTML和JPG
# 先儲存為HTML
html_file = 'trajectory_map.html'
m.save(html_file)
print(f"地圖已儲存為 {html_file}")

# 🌟 轉換為JPG圖片
# 方法1: 使用selenium (需要安裝: pip install selenium)
try:
    from selenium import webdriver
    from selenium.webdriver.chrome.options import Options
    import time
    import os
    
    # 設定Chrome選項
    chrome_options = Options()
    chrome_options.add_argument('--headless')  # 無頭模式
    chrome_options.add_argument('--no-sandbox')
    chrome_options.add_argument('--disable-dev-shm-usage')
    chrome_options.add_argument('--window-size=1080,1920')
    
    # 啟動瀏覽器
    driver = webdriver.Chrome(options=chrome_options)
    
    # 載入HTML檔案
    file_path = os.path.abspath(html_file)
    driver.get(f'file://{file_path}')
    
    # 等待地圖載入
    time.sleep(5)
    
    # 截圖儲存為JPG
    jpg_file = 'trajectory_map.jpg'
    driver.save_screenshot(jpg_file)
    driver.quit()
    
    print(f"地圖已儲存為 {jpg_file} (1920x1080)")

except ImportError:
    print("需要安裝selenium才能輸出JPG: pip install selenium")
    print("或使用方法2: imgkit")
    
    # 方法2: 使用imgkit (需要安裝: pip install imgkit 和 wkhtmltopdf)
    try:
        import imgkit
        
        # 將HTML轉換為JPG
        jpg_file = 'trajectory_map.jpg'
        imgkit.from_file(html_file, jpg_file, options={'format': 'jpg', 'width': 1920, 'height': 1080})
        print(f"地圖已儲存為 {jpg_file}")
        
    except ImportError:
        print("需要安裝imgkit才能使用方法2: pip install imgkit")
        print("並且需要安裝wkhtmltopdf: https://wkhtmltopdf.org/downloads.html")
        print("或者手動開啟 trajectory_map.html 並截圖儲存")

except Exception as e:
    print(f"截圖時發生錯誤: {e}")
    print("請手動開啟 trajectory_map.html 並截圖儲存")

# 顯示地圖
m

地圖已儲存為 trajectory_map.html
地圖已儲存為 trajectory_map.jpg (1920x1080)


In [9]:
import os
import re
import pandas as pd
import folium
import matplotlib
import matplotlib.colors as mcolors
import numpy as np
from folium.plugins import MeasureControl

def process_single_file(file_path, output_name):
    """處理單個檔案並生成地圖"""
    
    # 讀取檔案
    with open(file_path, 'r', encoding='utf-8') as f:
        lines = f.readlines()
    
    # Regex
    pattern = re.compile(
        r'^(?P<date>\d{2}-\d{2} \d{2}:\d{2}:\d{2}\.\d{3}).* \[(?P<object>.*?Variant\(Clone\))\]\s+location → '
        r'Lon: (?P<lon>[-\d\.]+)°, Lat: (?P<lat>[-\d\.]+)°, H: (?P<height>[\d\.]+) m'
    )
    destroy_pattern = re.compile(r"\[(.*?Variant\(Clone\))\].*Destroy")

    records = []
    destroy_events = []

    # 暫存每個物件最後座標
    last_position = {}

    for line in lines:
        m = pattern.search(line)
        if m:
            d = pd.to_datetime(m.group("date"), format="%m-%d %H:%M:%S.%f")
            obj = m.group("object")
            lon, lat, height = float(m.group("lon")), float(m.group("lat")), float(m.group("height"))

            records.append({"time": d, "object": obj, "lon": lon, "lat": lat, "height": height})

            # 更新最後座標
            last_position[obj] = (lat, lon)
        else:
            m2 = destroy_pattern.search(line)
            if m2:
                obj = m2.group(1)
                if obj in last_position:  # ✅ Destroy → 用前一筆座標
                    destroy_events.append({"object": obj, "lat": last_position[obj][0], "lon": last_position[obj][1]})

    # 如果沒有資料，返回None
    if not records:
        print(f"警告: {file_path} 沒有找到軌跡資料")
        return None

    df = pd.DataFrame(records).sort_values("time")
    # center = [df["lat"].mean(), df["lon"].mean()]  # 註解掉原來的中心點計算
    center = [51.541965, -0.014937]  # 固定中心點

    # 顏色與標籤
    object_colormaps = {
        "OneVariant(Clone)": "Blues",
        "TwoVariant(Clone)": "Reds",
        "ThreeVariant(Clone)": "Greens",
        "FourVariant(Clone)": "Oranges",
        "FiveVariant(Clone)": "Purples",
        "SixVariant(Clone)": "pink",
        "DebugCubeVariant(Clone)": "magma",
    }

    object_labels = {
        "OneVariant(Clone)": "1",
        "TwoVariant(Clone)": "2",
        "ThreeVariant(Clone)": "3",
        "FourVariant(Clone)": "4",
        "FiveVariant(Clone)": "5",
        "SixVariant(Clone)": "6",
        "DebugCubeVariant(Clone)": "D",
    }

    # ✅ 設定空底圖，讓我們可以自定義多種底圖
    m = folium.Map(location=center, zoom_start=16, tiles=None)

    # ✅ 衛星圖底圖（Esri World Imagery）
    folium.TileLayer(
        tiles="https://server.arcgisonline.com/ArcGIS/rest/services/World_Imagery/MapServer/tile/{z}/{y}/{x}",
        attr="Esri",
        name="Esri Satellite",
        overlay=False,
        control=True
    ).add_to(m)

    # ✅ CartoDB Dark Matter 底圖
    folium.TileLayer(
        tiles="https://{s}.basemaps.cartocdn.com/dark_all/{z}/{x}/{y}{r}.png",
        attr="CartoDB",
        name="CartoDB Dark Matter",
        overlay=False,
        control=True
    ).add_to(m)

    # ✅ CartoDB Positron 底圖
    folium.TileLayer(
        tiles="https://{s}.basemaps.cartocdn.com/light_all/{z}/{x}/{y}{r}.png",
        attr="CartoDB",
        name="CartoDB Positron",
        overlay=False,
        control=True
    ).add_to(m)

    # ✅ 指定順序：123456Debug
    object_order = [
        "OneVariant(Clone)",
        "TwoVariant(Clone)",
        "ThreeVariant(Clone)",
        "FourVariant(Clone)",
        "FiveVariant(Clone)",
        "SixVariant(Clone)",
        "DebugCubeVariant(Clone)",
    ]

    # 畫軌跡
    for obj in object_order:
        if obj not in df["object"].values and all(e["object"] != obj for e in destroy_events):
            continue  # 完全沒資料略過

        g = df[df["object"] == obj]
        trajectory = g[["lat", "lon"]].values.tolist()

        layer = folium.FeatureGroup(name=obj)

        # ✅ 畫軌跡（有兩點以上才畫線）
        if len(trajectory) > 1:
            cmap_name = object_colormaps.get(obj, "Greys")
            cmap_obj = matplotlib.colormaps[cmap_name]
            if cmap_name in ["pink", "Blues", "Reds", "Greens", "Oranges", "plasma"]:
                cmap = cmap_obj.reversed()  
            else:
                cmap = cmap_obj

            n_points = len(trajectory)
            
            # 🌟 改進的顏色與視覺效果設定 - 避免過淡的顏色
            for j in range(len(trajectory) - 1):
                # 🌟 顏色漸變：限制最小值避免過淡，從0.3到1.0的範圍
                color_intensity = 0.3 + (j / (n_points - 1)) * 0.7 if n_points > 1 else 0.8
                color = mcolors.to_hex(cmap(color_intensity))
                
                # 🌟 透明度固定為較高值，避免過於透明
                opacity = 0.8  # 固定透明度，不再漸變
                
                folium.PolyLine(
                    locations=[trajectory[j], trajectory[j + 1]],
                    color=color,
                    weight=4,  # 固定線寬
                    opacity=opacity,
                ).add_to(layer)

        # ✅ 起始與終點標記
        if len(g) > 0:
            first_point, last_point = g.iloc[0], g.iloc[-1]
            label = object_labels.get(obj, "?")

            if len(g) == 1:
                # 只有一點時，只顯示ending標記
                folium.Marker(
                    location=[first_point["lat"], first_point["lon"]],
                    popup=f"{obj} Point<br>Time: {first_point['time']}<br>Height: {first_point['height']:.1f}m",
                    icon=folium.DivIcon(
                        html=f'<div style="font-size:14px; color:white; background-color:red; '
                             f'border: 2px solid white; border-radius:50%; width:24px; height:24px; '
                             f'text-align:center; line-height:20px; font-weight:bold;">{label}</div>'
                    ),
                ).add_to(layer)
            else:
                # 多於一點時，顯示起始和終點標記
                # 🌟 改進起始點標記樣式
                folium.Marker(
                    location=[first_point["lat"], first_point["lon"]],
                    popup=f"{obj} Starting Point<br>Time: {first_point['time']}<br>Height: {first_point['height']:.1f}m",
                    icon=folium.DivIcon(
                        html=f'<div style="font-size:14px; color:white; background-color:gray; '
                             f'border: 2px solid white; border-radius:50%; width:24px; height:24px; '
                             f'text-align:center; line-height:20px; font-weight:bold;">{label}</div>'
                    ),
                ).add_to(layer)

                # 🌟 改進終點標記樣式
                folium.Marker(
                    location=[last_point["lat"], last_point["lon"]],
                    popup=f"{obj} Ending Point<br>Time: {last_point['time']}<br>Height: {last_point['height']:.1f}m",
                    icon=folium.DivIcon(
                        html=f'<div style="font-size:14px; color:white; background-color:red; '
                             f'border: 2px solid white; border-radius:50%; width:24px; height:24px; '
                             f'text-align:center; line-height:20px; font-weight:bold;">{label}</div>'
                    ),
                ).add_to(layer)

        # ✅ Destroy 標記（在上一筆座標）
        for e in destroy_events:
            if e["object"] == obj:
                label = object_labels.get(obj, "?")
                folium.Marker(
                    location=[e["lat"], e["lon"]],
                    popup=f"{obj} Destroy",
                    icon=folium.DivIcon(
                        html=f'<div style="font-size:14px; color:white; background-color:white; '
                             f'border: 2px solid white; border-radius:50%; width:24px; height:24px; '
                             f'text-align:center; line-height:20px; font-weight:bold;">🗑️</div>'
                    ),
                ).add_to(layer)

        layer.add_to(m)

    def dms_to_dd(d, m, s, ref):
        dd = d + m/60 + s/3600
        if ref in ['S', 'W']:
            dd *= -1
        return dd

    # 🔹 定義三組座標 (轉換後)

    OnePoolStreet_coords = [
        (dms_to_dd(51,32,18,'N'), dms_to_dd(0,0,36,'W')),
    ]

    one_points_coords = [
        (dms_to_dd(51,32,25,'N'), dms_to_dd(0,0,42,'W')),
    ]

    two_points_coords = [
        (dms_to_dd(51,32,14,'N'), dms_to_dd(0,1,5,'W')),
        (dms_to_dd(51,32,29,'N'), dms_to_dd(0,1,2,'W')),
    ]

    three_points_coords = [
        (dms_to_dd(51,32,47,'N'), dms_to_dd(0,0,53,'W')),
        (dms_to_dd(51,32,55,'N'), dms_to_dd(0,1,6,'W')),
    ]

    birds_coords = [
        (dms_to_dd(51,32,24,'N'), dms_to_dd(0,0,46,'W')),
        (dms_to_dd(51,32,16,'N'), dms_to_dd(0,0,52,'W')),
        (dms_to_dd(51,32,41,'N'), dms_to_dd(0,1,5,'W')),
        (dms_to_dd(51,32,59,'N'), dms_to_dd(0,1,16,'W')),
    ]

    bike_coords = [
        (dms_to_dd(51,32,18,'N'), dms_to_dd(0,0,42,'W')),
        (dms_to_dd(51,32,29,'N'), dms_to_dd(0,0,36,'W')),
        (dms_to_dd(51,32,26,'N'), dms_to_dd(0,1,15,'W')),
    ]

    # 🔹 建立三個圖層
    points_layer = folium.FeatureGroup(name="📍 Points")
    birds_layer = folium.FeatureGroup(name="🐦 Birds") 
    bike_layer = folium.FeatureGroup(name="🚴 Bikes")

    # 🔹 points

    for i, (lat, lon) in enumerate(OnePoolStreet_coords, 1):
        folium.Marker(
            location=[lat, lon],
            popup=f"Point {i}",
            icon=folium.Icon(color="darkred", icon="home"),
        ).add_to(points_layer)

    
    for i, (lat, lon) in enumerate(one_points_coords, 1):
        folium.Marker(
            location=[lat, lon],
            popup=f"Point {i}",
            icon=folium.Icon(color="lightgreen", icon="map-marker"),
        ).add_to(points_layer)

        # 🔹 points
    for i, (lat, lon) in enumerate(two_points_coords, 1):
        folium.Marker(
            location=[lat, lon],
            popup=f"Point {i}",
            icon=folium.Icon(color="green", icon="map-marker"),
        ).add_to(points_layer)

        # 🔹 points
    for i, (lat, lon) in enumerate(three_points_coords, 1):
        folium.Marker(
            location=[lat, lon],
            popup=f"Point {i}",
            icon=folium.Icon(color="darkgreen", icon="map-marker"),
        ).add_to(points_layer)

    # 🔹 birds
    for i, (lat, lon) in enumerate(birds_coords, 1):
        folium.Marker(
            location=[lat, lon],
            popup=f"Bird {i}",
            icon=folium.Icon(color="orange", icon="star", prefix="fa"),
        ).add_to(birds_layer)

    # 🔹 bike
    for i, (lat, lon) in enumerate(bike_coords, 1):
        folium.Marker(
            location=[lat, lon],
            popup=f"Bike {i}",
            icon=folium.Icon(color="blue", icon="bicycle", prefix="fa"),
        ).add_to(bike_layer)

    # 🔹 把圖層加到地圖
    points_layer.add_to(m)
    birds_layer.add_to(m)
    bike_layer.add_to(m)

    # # 🔹 更新圖層控制
    # folium.LayerControl(collapsed=False).add_to(m)

    # 🌟 加入比例尺
    # MeasureControl().add_to(m)

    # 🌟 加入詳細的比例尺顯示
    scale_html = '''
    <div style="position: fixed; 
         bottom: 30px; right: 30px; width: 300px; height: 80px; 
         background-color: rgba(255,255,255,0.9); border: 2px solid black; z-index:9999; 
         font-size: 14px; padding: 15px; font-family: Arial;">
    <div style="text-align: center; font-weight: bold; margin-bottom: 0px; font-size: 16px;">Scale</div>

    <!-- 主要比例尺 -->
    <div style="position: relative; height: 35px; margin-bottom: 0px;">
      <div style="border-bottom: 4px solid black; width: 250px; position: absolute; bottom: 15px;"></div>
      <div style="border-left: 3px solid black; height: 12px; position: absolute; left: 0; bottom: 15px;"></div>
      <div style="border-left: 3px solid black; height: 12px; position: absolute; left: 62.5px; bottom: 15px;"></div>
      <div style="border-left: 3px solid black; height: 12px; position: absolute; left: 125px; bottom: 15px;"></div>
      <div style="border-left: 3px solid black; height: 12px; position: absolute; left: 187.5px; bottom: 15px;"></div>
      <div style="border-left: 3px solid black; height: 12px; position: absolute; left: 250px; bottom: 15px;"></div>
      
      <!-- 標籤移到比例尺上方 -->
      <span style="position: absolute; left: -5px; bottom: 0px; font-size: 12px; font-weight: bold;">0</span>
      <span style="position: absolute; left: 45px; bottom: 0px; font-size: 12px; font-weight: bold;">25m</span>
      <span style="position: absolute; left: 105px; bottom: 0px; font-size: 12px; font-weight: bold;">50m</span>
      <span style="position: absolute; left: 165px; bottom: 0px; font-size: 12px; font-weight: bold;">75m</span>
      <span style="position: absolute; left: 225px; bottom: 0px; font-size: 12px; font-weight: bold;">100m</span>
    </div>
    </div>
    '''
    m.get_root().html.add_child(folium.Element(scale_html))


    # 🌟 新增圖例說明 - 移到比例尺上方
    legend_html = '''
    <div style="position: fixed; 
         bottom: 130px; right: 30px; width: 220px; height: 250px; 
         background-color: white; border: 2px solid grey; z-index:9999; 
         font-size: 12px; padding: 10px;">
    <p><strong>Legend</strong></p>
    <p>⚫ Starting | 🔴 Ending</p>
    <p>🗑️ Destroy</p>
    <p>Line：bright → dark </p>
    <p>by time sequence</p>
    <hr style="margin: 8px 0;">
    <p><span style="color: darkred;">🏠</span> Starting and ending point</p>
    <p><span style="color: blue;">🚴</span> Share bike location</p>
    <p><span style="color: orange;">⭐</span> Bird observation</p>
    </div>
    '''
    m.get_root().html.add_child(folium.Element(legend_html))


    return m

def batch_process_files():
    """批量處理所有檔案"""
    
    folder_path = "all testing progress"
    
    # 檢查資料夾是否存在
    if not os.path.exists(folder_path):
        print(f"錯誤: 找不到資料夾 '{folder_path}'")
        return
    
    # 處理 1.txt 到 18.txt
    for i in range(1, 19):
        txt_file = os.path.join(folder_path, f"{i}.txt")
        
        if not os.path.exists(txt_file):
            print(f"警告: 找不到檔案 {txt_file}")
            continue
        
        print(f"正在處理 {txt_file}...")
        
        try:
            # 生成地圖
            map_obj = process_single_file(txt_file, f"{i}")
            
            if map_obj is None:
                print(f"跳過 {txt_file} (無軌跡資料)")
                continue
            
            # 儲存為HTML
            html_file = f"trajectory_map_{i}.html"
            map_obj.save(html_file)
            print(f"  └─ 已儲存 {html_file}")
            
            # 轉換為JPG
            jpg_file = f"{i}.jpg"
            
            try:
                from selenium import webdriver
                from selenium.webdriver.chrome.options import Options
                import time
                
                # 設定Chrome選項
                chrome_options = Options()
                chrome_options.add_argument('--headless')
                chrome_options.add_argument('--no-sandbox')
                chrome_options.add_argument('--disable-dev-shm-usage')
                chrome_options.add_argument('--window-size=1200,1820')
                
                # 啟動瀏覽器
                driver = webdriver.Chrome(options=chrome_options)
                
                # 載入HTML檔案
                file_path = os.path.abspath(html_file)
                driver.get(f'file://{file_path}')
                
                # 等待地圖載入
                time.sleep(5)
                
                # 截圖儲存為JPG
                driver.save_screenshot(jpg_file)
                driver.quit()
                
                print(f"  └─ 已儲存 {jpg_file}")
                
                # 清理HTML檔案（可選）
                # os.remove(html_file)
                
            except ImportError:
                print(f"  └─ 需要安裝selenium: pip install selenium")
                print(f"  └─ 已儲存HTML檔案，請手動截圖: {html_file}")
                
            except Exception as e:
                print(f"  └─ 截圖時發生錯誤: {e}")
                print(f"  └─ 已儲存HTML檔案，請手動截圖: {html_file}")
        
        except Exception as e:
            print(f"處理 {txt_file} 時發生錯誤: {e}")
            continue
    
    print("批量處理完成！")

# 執行批量處理
if __name__ == "__main__":
    batch_process_files()

正在處理 all testing progress\1.txt...
  └─ 已儲存 trajectory_map_1.html


C:\Users\PeggyTsai\anaconda3\Lib\site-packages\selenium\webdriver\remote\webdriver.py:843: UserWarning: name used for saved screenshot does not match file type. It should end with a `.png` extension
  return self.get_screenshot_as_file(filename)


  └─ 已儲存 1.jpg
正在處理 all testing progress\2.txt...
  └─ 已儲存 trajectory_map_2.html


C:\Users\PeggyTsai\anaconda3\Lib\site-packages\selenium\webdriver\remote\webdriver.py:843: UserWarning: name used for saved screenshot does not match file type. It should end with a `.png` extension
  return self.get_screenshot_as_file(filename)


  └─ 已儲存 2.jpg
正在處理 all testing progress\3.txt...
  └─ 已儲存 trajectory_map_3.html


...

  └─ 已儲存 16.jpg
正在處理 all testing progress\17.txt...
  └─ 已儲存 trajectory_map_17.html


C:\Users\PeggyTsai\anaconda3\Lib\site-packages\selenium\webdriver\remote\webdriver.py:843: UserWarning: name used for saved screenshot does not match file type. It should end with a `.png` extension
  return self.get_screenshot_as_file(filename)


  └─ 已儲存 17.jpg
正在處理 all testing progress\18.txt...
  └─ 已儲存 trajectory_map_18.html


C:\Users\PeggyTsai\anaconda3\Lib\site-packages\selenium\webdriver\remote\webdriver.py:843: UserWarning: name used for saved screenshot does not match file type. It should end with a `.png` extension
  return self.get_screenshot_as_file(filename)


  └─ 已儲存 18.jpg
批量處理完成！
